---
title: "Advanced GWAS — Python Worked Example"
author: "Nivedita Bhadra"
date: 2026-07-02
page-layout: article
format:
  html:
    toc: false
    code-copy: true
jupyter: python3
execute:
  echo: true
---

This notebook simulates genotype data, constructs a GRM, computes principal components, and compares association tests with and without PC adjustment. It is deliberately small so it runs quickly on a laptop.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
np.random.seed(1)

In [ ]:
# Simulate genotypes (N x M)
N = 800  # individuals
M = 2000 # SNPs (kept small)
p = np.random.uniform(0.05, 0.5, size=M)
G = np.random.binomial(2, p, size=(N, M)).astype(float)
# Introduce one causal SNP (index 0) with modest effect
beta = np.zeros(M)
beta[0] = 0.6
# Simulate phenotype: polygenic background + causal SNP + noise
polygenic = G @ (np.random.normal(0, 0.01, size=M))
pheno = G[:, 0] * beta[0] + polygenic + np.random.normal(0, 1.0, size=N)

In [ ]:
# Construct standardized genotype matrix Z and GRM K = Z Z^T / M
p_hat = G.mean(axis=0) / 2.0
std = np.sqrt(2 * p_hat * (1 - p_hat))
# avoid division by zero
std[std == 0] = 1.0
Z = (G - 2 * p_hat) / std
K = (Z @ Z.T) / M
print('K shape', K.shape)

In [ ]:
# Compute top PCs from K (eigendecomposition)
eigvals, eigvecs = np.linalg.eigh(K)
idx = np.argsort(eigvals)[::-1]
pcs = eigvecs[:, idx][:, :10]  # top 10 PCs
print('Top eigenvalues:', eigvals[idx][:5])

In [ ]:
# Simple OLS function to get beta, se, t, p
def ols_pval(X, y):
    X = np.asarray(X)
    y = np.asarray(y)
    n, p = X.shape
    XtX_inv = np.linalg.inv(X.T @ X)
    beta_hat = XtX_inv @ X.T @ y
    resid = y - X @ beta_hat
    sigma2 = (resid @ resid) / (n - p)
    se = np.sqrt(np.diag(XtX_inv) * sigma2)
    tstats = beta_hat / se
    pvals = 2 * stats.t.sf(np.abs(tstats), df=n - p)
    return beta_hat, se, tstats, pvals

# Test association for SNP 0 with and without PCs
X_null = np.column_stack([np.ones(N)])
X_snp = np.column_stack([np.ones(N), G[:, 0]])
X_snp_pcs = np.column_stack([np.ones(N), G[:, 0], pcs])
b0, se0, t0, p0 = ols_pval(X_snp, pheno)
b1, se1, t1, p1 = ols_pval(X_snp_pcs, pheno)
print('SNP0 p-value without PCs:', p0[1])
print('SNP0 p-value with PCs:', p1[1])

In [ ]:
# Genome-wide scan (simple per-SNP regression) to build QQ and lambda_GC
pvals = np.empty(M)
for j in range(M):
    Xj = np.column_stack([np.ones(N), G[:, j]])
    _, _, _, pj = ols_pval(Xj, pheno)
    pvals[j] = pj[1]
# compute lambda_GC
chisq = stats.chi2.isf(pvals, df=1)
lambda_gc = np.median(chisq) / 0.4549364
print('lambda_GC:', lambda_gc)

# QQ plot
expected = -np.log10(np.linspace(1/M, 1, M))
observed = -np.log10(np.sort(pvals))
plt.figure(figsize=(5,5))
plt.plot(expected, observed, marker='.', linestyle='none')
plt.plot([0, max(expected)], [0, max(expected)], color='grey')
plt.xlabel('Expected -log10(p)')
plt.ylabel('Observed -log10(p)')
plt.title('QQ plot (simple scan)')
plt.show()

Summary:
- The notebook simulates a small GWAS, constructs a GRM, and computes PCs.
- It compares the SNP association p-value with and without PC adjustment and produces a QQ plot and lambda_GC.
- For production analyses, replace the simple per-SNP OLS with mixed-model software (BOLT-LMM, GCTA fastGWA, REGENIE) and use LD score regression for partitioning inflation.